# Devanagari End-to-End OCR Classification & Reconstruction Pipeline
### Self-Contained 3-Zone Architecture with Hierarchical Cascade Multi-Model Inference

This notebook implements the complete, self-contained end-to-end Devanagari OCR pipeline:
1. **Preprocessing & Binarization**: Grayscale adaptive thresholding and border padding.
2. **Shirorekha Removal**: Horizontal profile analysis, headline strip, and vertical stem reconnection.
3. **3-Zone Segmentation**: Dynamic division into Upper (ascenders), Middle (core characters), and Lower (descenders).
4. **Hierarchical Cascade Multi-Model Inference**:
   - **Upper Modifier Model** (11 classes): Vowel ascenders, bindu, and reph (`upper_modifier.pth`).
   - **Lower Modifier Model** (7 classes): Vowel descenders, halant, and rakar (`lower_modifier.pth`).
   - **Full Character Model** (59 classes): Primary classifier for consonants, vowels, digits (`full_character.pth`).
   - **Half Character Model** (26 classes): Fallback for stemless half-consonants (`half_character.pth`).
   - **Compound Character Model** (21 classes): Fallback for fused/stacked ligatures (`compound_character.pth`).
5. **Unicode Word Assembler**: Phonetic Devanagari ordering (reph prepending, chhoti-i realignment, matras).
6. **Validation & Metrics**: Evaluates Character Accuracy %, Character Error Rate (CER), and Word Exact Match on `test.parquet`.


In [ ]:
import os
import time
import unicodedata
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import json
import torch
import torch.nn as nn
from torchvision import transforms

# Set compute device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch Version:", torch.__version__)
print("Using Compute Device:", device)

# Load native font for Devanagari plot titles (suppresses missing glyph warnings)
FONT_PATH = "model/fonts/NotoSans-Regular.ttf"
dev_font = fm.FontProperties(fname=FONT_PATH) if os.path.exists(FONT_PATH) else None


## 1. Model Architecture & Loading
All 5 specialized models share the identical 3-stage CNN architecture (64×64 single-channel, normalized ink = 1.0).


In [ ]:
class CNN3Stage(nn.Module):
    """3-stage CNN for 64x64 Devanagari glyph recognition."""
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.10),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.20),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.30),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.40),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def load_model_and_classes(folder_name, weight_file):
    base = os.path.join("model", folder_name)
    ckpt_path = os.path.join(base, weight_file)
    class_file = os.path.join(base, "classes.json")
    
    with open(class_file, encoding="utf-8") as f:
        class_meta = json.load(f)

    # Safely map both folder name and class name to glyph (supporting 'sign' or 'char')
    char_map = {}
    for item in class_meta:
        glyph = item.get("sign") or item.get("char") or ""
        char_map[item["name"]] = glyph
        if "folder" in item:
            char_map[item["folder"]] = glyph

    ckpt = torch.load(ckpt_path, map_location=device)
    classes = ckpt["classes"]
    model = CNN3Stage(ckpt["num_classes"]).to(device).eval()
    model.load_state_dict(ckpt["model_state_dict"])
    return model, classes, char_map


# Load all 5 models
full_model, full_classes, full_map = load_model_and_classes("full_character", "full_character.pth")
half_model, half_classes, half_map = load_model_and_classes("half_character", "half_character.pth")
comp_model, comp_classes, comp_map = load_model_and_classes("compound_character", "compound_character.pth")
lower_model, lower_classes, lower_map = load_model_and_classes("lower_modifier", "lower_modifier.pth")
upper_model, upper_classes, upper_map = load_model_and_classes("upper_modifier", "upper_modifier.pth")

print("✓ All 5 models loaded successfully:")
print(f"  • Full Character Model:     {len(full_classes)} classes (Consonants, Vowels, Digits)")
print(f"  • Half Character Model:     {len(half_classes)} classes (Stemless Half-Consonants)")
print(f"  • Compound Character Model: {len(comp_classes)} classes (Fused/Stacked Conjuncts)")
print(f"  • Lower Modifier Model:     {len(lower_classes)} classes (Descenders & Diacritics)")
print(f"  • Upper Modifier Model:     {len(upper_classes)} classes (Ascenders & Reph)")


## 2. Image Preprocessing & 3-Zone Segmentation
- Removes Shirorekha (headline) band while reconnecting vertical stems.
- Divides components into Upper Zone ($y < y_{\text{headline}}$), Middle Zone, and Lower Zone ($y > y_{\text{baseline}}$).


In [ ]:
REF_HEIGHT = 34
CANVAS_SIZE = 64

def normalize_crop_to_64(gray_crop, base_height=REF_HEIGHT):
    """Scales segmented crop to match the 64x64 training scale."""
    scale = REF_HEIGHT / float(max(1, base_height))
    h, w = gray_crop.shape
    new_h = max(1, round(h * scale))
    new_w = max(1, round(w * scale))

    scaled = cv2.resize(gray_crop, (new_w, new_h), interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC)
    if max(new_h, new_w) > CANVAS_SIZE - 4:
        shrink = (CANVAS_SIZE - 4) / max(new_h, new_w)
        scaled = cv2.resize(scaled, (max(1, round(new_w * shrink)), max(1, round(new_h * shrink))), interpolation=cv2.INTER_AREA)

    canvas = np.full((CANVAS_SIZE, CANVAS_SIZE), 255, dtype=np.uint8)
    y_off = (CANVAS_SIZE - scaled.shape[0]) // 2
    x_off = (CANVAS_SIZE - scaled.shape[1]) // 2
    canvas[y_off : y_off + scaled.shape[0], x_off : x_off + scaled.shape[1]] = scaled

    # Convert to inverted float tensor: ink = 1.0, background = 0.0
    tensor = (1.0 - torch.from_numpy(canvas).float() / 255.0).unsqueeze(0).unsqueeze(0).to(device)
    return tensor


def segment_word_image(input_img):
    """Segments a word image into Upper, Middle, and Lower zone components."""
    if len(input_img.shape) == 3:
        gray = cv2.cvtColor(input_img, cv2.COLOR_BGR2GRAY)
    else:
        gray = input_img.copy()

    # Adaptive binarization
    bin_img = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 21, 2)
    if np.sum(bin_img == 0) > np.sum(bin_img == 255):
        bin_img = cv2.bitwise_not(bin_img)
    padded = cv2.copyMakeBorder(bin_img, 4, 4, 4, 4, cv2.BORDER_CONSTANT, value=255)

    # Shirorekha detection
    binary_sh = (padded == 0).astype("uint8")
    h_profile = np.sum(binary_sh, axis=1)
    max_h = np.max(h_profile)
    if max_h == 0:
        return {"upper": [], "middle": [], "lower": [], "base_height": 28, "vis": padded}

    threshold = 0.65 * max_h
    rows = np.where(h_profile >= threshold)[0]
    valid_bands = []
    if len(rows) > 0:
        bands = [[rows[0]]]
        for r in rows[1:]:
            if r == bands[-1][-1] + 1:
                bands[-1].append(r)
            else:
                bands.append([r])
        w = padded.shape[1]
        valid_bands = [b for b in bands if np.mean([np.sum(padded[r] == 0) / w for r in b]) > 0.35]

    # Remove headline
    sh_removed = padded.copy()
    for b in valid_bands:
        mid = int(np.mean(b))
        for t in range(-1, 2):
            if 0 <= mid + t < sh_removed.shape[0]:
                sh_removed[mid + t, :] = 255

    # Vertical reconnection
    reconnected = sh_removed.copy()
    if valid_bands:
        cut_rows = [int(np.mean(b)) for b in valid_bands]
        z_top, z_bottom = max(0, min(cut_rows) - 3), min(padded.shape[0], max(cut_rows) + 3)
        v_prof = np.sum((sh_removed == 0).astype("uint8"), axis=0)
        max_vp = np.max(v_prof)
        if max_vp > 0:
            for col in np.where(v_prof >= 0.70 * max_vp)[0]:
                r = z_top
                while r < z_bottom:
                    if reconnected[r, col] == 0:
                        r2 = r + 1
                        gap = 0
                        while r2 < z_bottom and reconnected[r2, col] == 255:
                            gap += 1
                            if gap > 5: break
                            r2 += 1
                        if r2 < z_bottom and 0 < gap <= 5:
                            reconnected[r + 1 : r2, col] = 0
                            r = r2
                        else: r += 1
                    else: r += 1

    # 3-Zone Connected Components
    binary = (reconnected == 0).astype("uint8")
    num_labels_init, _, stats_init, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    if num_labels_init <= 1:
        return {"upper": [], "middle": [], "lower": [], "base_height": 28, "vis": reconnected}

    heights = stats_init[1:, cv2.CC_STAT_HEIGHT]
    centers_y = stats_init[1:, cv2.CC_STAT_TOP] + (heights / 2.0)
    med_h = float(np.median(heights))
    med_c = float(np.median(centers_y))
    mid_top = int(med_c - 0.55 * med_h)
    mid_bottom = int(med_c + 0.55 * med_h)

    # Split descenders extending into lower zone
    split_binary = binary.copy()
    margin = max(2, int(med_h * 0.20))
    for p in range(1, num_labels_init):
        x, y, w_comp, h = stats_init[p, :4]
        if y < mid_bottom and (y + h) > (mid_bottom + margin):
            roi = split_binary[y : y + h, x : x + w_comp]
            loc_mid = mid_bottom - y
            s_start, s_end = max(0, loc_mid - margin), min(roi.shape[0], loc_mid + margin)
            if s_start < s_end:
                r_sums = np.sum(roi[s_start:s_end, :], axis=1)
                if len(r_sums) > 0:
                    split_y = y + s_start + int(np.argmin(r_sums))
                    split_binary[max(0, split_y - 1) : split_y, x : x + w_comp] = 0

    num_labels, _, stats, centroids = cv2.connectedComponentsWithStats(split_binary, connectivity=8)
    categorized = {"upper": [], "middle": [], "lower": []}

    for p in range(1, num_labels):
        x, y, w_comp, h, area = stats[p]
        if area < 5: continue
        cy = y + (h / 2.0)
        zone = "upper" if cy < mid_top else ("lower" if cy > mid_bottom else "middle")

        crop = reconnected[max(0, y - 1) : y + h + 1, max(0, x - 1) : x + w_comp + 1]
        categorized[zone].append({
            "x": int(x), "y": int(y), "w": int(w_comp), "h": int(h),
            "cx": float(centroids[p][0]), "crop": crop,
            "aspect_ratio": float(w_comp) / max(1.0, float(h))
        })

    for z in categorized:
        categorized[z].sort(key=lambda item: item["x"])

    m_heights = [c["h"] for c in categorized["middle"]]
    base_h = float(np.median(m_heights)) if m_heights else med_h
    return {"upper": categorized["upper"], "middle": categorized["middle"], "lower": categorized["lower"], "base_height": base_h, "vis": reconnected}


## 3. Hierarchical Cascade Inference & Unicode Reconstruction
- **Cascade Routing**: Queries Full-Character model first. If confidence $\ge 0.82$, accepts it; otherwise falls back to Half-Character / Compound CNNs.
- **Syllable Association & Grammar**: Binds diacritics to consonants, prepends reph (`र्`), realigns chhoti-i (`ि`), and normalizes Unicode (NFC).


In [ ]:
@torch.no_grad()
def predict_glyph(model, classes, char_map, crop, base_height):
    x = normalize_crop_to_64(crop, base_height)
    probs = torch.softmax(model(x), dim=1)[0]
    idx = torch.argmax(probs).item()
    cls_name = classes[idx]
    return {"class": cls_name, "text": char_map.get(cls_name, ""), "conf": probs[idx].item()}


@torch.no_grad()
def route_middle_component(comp, base_height, threshold=0.82):
    crop = comp["crop"]
    w, h = comp["w"], comp["h"]

    # Check for isolated vertical stem (aa-matra 'ा')
    if w <= 4 and h >= 0.65 * base_height:
        return {"type": "stem", "text": "ा", "conf": 0.99, "strategy": "stem"}

    # Primary pass: Full Character Model
    pred_full = predict_glyph(full_model, full_classes, full_map, crop, base_height)
    if pred_full["conf"] >= threshold:
        pred_full.update({"type": "full", "strategy": "cascade_pass"})
        return pred_full

    # Fallback pass: Half & Compound models
    pred_half = predict_glyph(half_model, half_classes, half_map, crop, base_height)
    pred_comp = predict_glyph(comp_model, comp_classes, comp_map, crop, base_height)

    if comp["aspect_ratio"] < 0.70 and pred_half["conf"] >= 0.35:
        pred_half.update({"type": "half", "strategy": "fallback_half"})
        return pred_half

    if pred_comp["conf"] > pred_half["conf"]:
        pred_comp.update({"type": "compound", "strategy": "fallback_compound"})
        return pred_comp
    else:
        pred_half.update({"type": "half", "strategy": "fallback_half"})
        return pred_half


def reconstruct_word_text(mid_preds, up_preds, low_preds):
    """Assembles recognized components into canonical Unicode Devanagari text."""
    if not mid_preds:
        return ""

    syllables = [{"mid": m, "upper": [], "lower": [], "cx": m["comp"]["cx"]} for m in mid_preds]

    # Associate Upper Modifiers
    for u in up_preds:
        u_cx = u["comp"]["cx"]
        if "i_matra" in u["class"]:  # chhoti-i extends leftwards
            idx = 0
            best_dist = 9999
            for i, syl in enumerate(syllables):
                d = syl["cx"] - u_cx
                if -5 <= d < best_dist:
                    best_dist = d
                    idx = i
            syllables[idx]["upper"].append(u)
        else:
            idx = min(range(len(syllables)), key=lambda i: abs(syllables[i]["cx"] - u_cx))
            syllables[idx]["upper"].append(u)

    # Associate Lower Modifiers
    for l in low_preds:
        idx = min(range(len(syllables)), key=lambda i: abs(syllables[i]["cx"] - l["comp"]["cx"]))
        syllables[idx]["lower"].append(l)

    word = ""
    for syl in syllables:
        m = syl["mid"]
        if m["type"] == "stem":
            if word and not word.endswith("ा"):
                word += "ा"
            else:
                word += m["text"]
            continue

        chunk = ""
        # Prepend Reph (र्)
        reph = any("reph" in u["class"] for u in syl["upper"])
        if reph: chunk += "र्"

        # Base Consonant / Conjunct
        chunk += m["text"]

        # Lower Modifiers
        for l in syl["lower"]:
            chunk += "़" if "nukta" in l["class"] else ("्र" if "rakar" in l["class"] else l["text"])

        # Upper Modifiers (Vowels, then Anusvara/Chandrabindu)
        non_reph = [u for u in syl["upper"] if "reph" not in u["class"]]
        for u in non_reph:
            if "anusvara" not in u["class"] and "chandrabindu" not in u["class"]:
                chunk += u["text"]
        for u in non_reph:
            if "anusvara" in u["class"] or "chandrabindu" in u["class"]:
                chunk += u["text"]

        word += chunk

    return unicodedata.normalize("NFC", word)


def recognize_word(img):
    start = time.time()
    seg = segment_word_image(img)
    base_h = seg["base_height"]

    up_preds = []
    for c in seg["upper"]:
        p = predict_glyph(upper_model, upper_classes, upper_map, c["crop"], base_h)
        p["comp"] = c
        up_preds.append(p)

    low_preds = []
    for c in seg["lower"]:
        p = predict_glyph(lower_model, lower_classes, lower_map, c["crop"], base_h)
        p["comp"] = c
        low_preds.append(p)

    mid_preds = []
    for c in seg["middle"]:
        p = route_middle_component(c, base_h)
        p["comp"] = c
        mid_preds.append(p)

    text = reconstruct_word_text(mid_preds, up_preds, low_preds)
    latency = time.time() - start
    return {"text": text, "latency": latency, "seg": seg, "mid_preds": mid_preds}


## 4. Single-Word Pipeline Demonstration
Tests the complete pipeline on a real Sanskrit word from `test.parquet`.


In [ ]:
def levenshtein(s1, s2):
    s1, s2 = unicodedata.normalize("NFC", s1.strip()), unicodedata.normalize("NFC", s2.strip())
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if s1[i-1] == s2[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp[m][n]

def compute_cer(ref, hyp):
    ref_norm = unicodedata.normalize("NFC", ref.strip())
    if not ref_norm: return 0.0 if not hyp else 1.0
    return levenshtein(ref_norm, hyp) / float(len(ref_norm))


# Load test dataset
df_test = pd.read_parquet("dataset/data/test.parquet")
sample_row = df_test.iloc[1]
raw = sample_row.get("image.bytes", sample_row.get("image_bytes"))
nparr = np.frombuffer(raw, np.uint8)
img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

# Run Recognition
res = recognize_word(img)
gt = sample_row["label"]
pred = res["text"]
cer = compute_cer(gt, pred)
acc = max(0.0, 1.0 - cer) * 100.0

print(f"Ground Truth:       '{gt}'")
print(f"Predicted Text:     '{pred}'")
print(f"Character Accuracy: {acc:.1f}%")
print(f"CER:                {cer:.3f}")
print(f"Latency:            {res['latency'] * 1000.0:.1f} ms")

# Plot 3-zone color-coded bounding boxes
vis = cv2.cvtColor(res["seg"]["vis"], cv2.COLOR_GRAY2RGB)
for c in res["seg"]["upper"]:  cv2.rectangle(vis, (c["x"], c["y"]), (c["x"] + c["w"], c["y"] + c["h"]), (0, 100, 255), 1)
for c in res["seg"]["middle"]: cv2.rectangle(vis, (c["x"], c["y"]), (c["x"] + c["w"], c["y"] + c["h"]), (255, 30, 30), 1)
for c in res["seg"]["lower"]:  cv2.rectangle(vis, (c["x"], c["y"]), (c["x"] + c["w"], c["y"] + c["h"]), (30, 200, 30), 1)

fig, ax = plt.subplots(figsize=(10, 2.8))
ax.imshow(vis)
title_text = f"GT: {gt}  |  Pred: {pred}  |  Acc: {acc:.0f}%"
ax.set_title(title_text, fontproperties=dev_font, fontsize=12) if dev_font else ax.set_title(title_text, fontsize=12)
ax.axis("off")
plt.tight_layout()
plt.show()


## 5. Quantitative Benchmark on Test Dataset Batch
Runs evaluation on a batch of 30 Sanskrit test words.


In [ ]:
BATCH_SIZE = 30
records = []

for i in range(min(BATCH_SIZE, len(df_test))):
    row = df_test.iloc[i]
    raw = row.get("image.bytes", row.get("image_bytes"))
    nparr = np.frombuffer(raw, np.uint8)
    sample_img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    
    out = recognize_word(sample_img)
    lbl = row["label"]
    p_text = out["text"]
    c_err = compute_cer(lbl, p_text)
    
    records.append({
        "filename": row.get("filename", f"sample_{i}.png"),
        "ground_truth": lbl,
        "prediction": p_text,
        "char_accuracy": max(0.0, 1.0 - c_err) * 100.0,
        "cer": c_err,
        "exact_match": 1.0 if unicodedata.normalize("NFC", lbl) == p_text else 0.0,
        "latency_ms": out["latency"] * 1000.0,
    })

eval_df = pd.DataFrame(records)

print("=" * 50)
print("          QUANTITATIVE EVALUATION SUMMARY")
print("=" * 50)
print(f"  Samples Evaluated:       {len(eval_df)}")
print(f"  Mean Character Accuracy: {eval_df['char_accuracy'].mean():.1f}%")
print(f"  Mean CER:                {eval_df['cer'].mean():.3f}")
print(f"  Exact Word Match:        {eval_df['exact_match'].mean() * 100.0:.1f}%")
print(f"  Average Latency:         {eval_df['latency_ms'].mean():.1f} ms / word")
print("=" * 50)


## 6. Detailed Results Table


In [ ]:
results_table = pd.DataFrame({
    "Filename": eval_df["filename"],
    "Ground Truth": eval_df["ground_truth"],
    "Prediction": eval_df["prediction"],
    "Char Acc": eval_df["char_accuracy"].round(1).astype(str) + "%",
    "CER": eval_df["cer"].round(2),
    "Exact": eval_df["exact_match"].apply(lambda m: "✓" if m == 1.0 else "✗"),
    "Latency (ms)": eval_df["latency_ms"].round(1),
})
results_table.head(20)


## 7. Visualizing Predictions Across Multiple Test Words


In [ ]:
for idx in [0, 2, 4, 6]:
    if idx < len(df_test):
        row = df_test.iloc[idx]
        raw = row.get("image.bytes", row.get("image_bytes"))
        nparr = np.frombuffer(raw, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        res = recognize_word(img)
        gt_val = row["label"]
        p_val = res["text"]
        c_val = compute_cer(gt_val, p_val)
        a_val = max(0.0, 1.0 - c_val) * 100.0

        vis_box = cv2.cvtColor(res["seg"]["vis"], cv2.COLOR_GRAY2RGB)
        for c in res["seg"]["upper"]:  cv2.rectangle(vis_box, (c["x"], c["y"]), (c["x"] + c["w"], c["y"] + c["h"]), (0, 100, 255), 1)
        for c in res["seg"]["middle"]: cv2.rectangle(vis_box, (c["x"], c["y"]), (c["x"] + c["w"], c["y"] + c["h"]), (255, 30, 30), 1)
        for c in res["seg"]["lower"]:  cv2.rectangle(vis_box, (c["x"], c["y"]), (c["x"] + c["w"], c["y"] + c["h"]), (30, 200, 30), 1)

        fig, ax = plt.subplots(figsize=(10, 2.6))
        ax.imshow(vis_box)
        t_str = f"Sample #{idx} '{row['filename']}' | GT: {gt_val} | Pred: {p_val} | Acc: {a_val:.0f}%"
        ax.set_title(t_str, fontproperties=dev_font, fontsize=11) if dev_font else ax.set_title(t_str, fontsize=11)
        ax.axis("off")
        plt.tight_layout()
        plt.show()
